# Noncommutativity grid analysis

This notebook evaluates the empirical Sobol-profile noncommutativity metrics from
`noncommutativity_detailed.tex` across benchmark/transformation pairs that are
compatible with the current `sabench` registries.

The notebook is intentionally registry-driven. It does not maintain a manual list
of benchmark or transform keys; by default it evaluates the live benchmark and
transform registries with a small, deterministic Monte Carlo design. Environment
variables can shrink the grid for CI or fast smoke execution.

## Configuration

The default configuration evaluates all registered benchmark/transformation pairs
with a small sample size. Set `SABENCH_GRID_MAX_BENCHMARKS`,
`SABENCH_GRID_MAX_TRANSFORMS`, or `SABENCH_GRID_N_BASE` in the environment to run
a smaller smoke grid.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from sabench.analysis.grid import evaluate_noncommutativity_grid
from sabench.benchmarks import BENCHMARK_REGISTRY
from sabench.transforms import TRANSFORM_REGISTRY

N_BASE = int(os.environ.get("SABENCH_GRID_N_BASE", "128"))
RNG_SEED = int(os.environ.get("SABENCH_GRID_SEED", "20260429"))
TAU = float(os.environ.get("SABENCH_GRID_TAU", "0.05"))
TOP_K = int(os.environ.get("SABENCH_GRID_TOP_K", "3"))
MAX_BENCHMARKS = int(os.environ.get("SABENCH_GRID_MAX_BENCHMARKS", "0"))
MAX_TRANSFORMS = int(os.environ.get("SABENCH_GRID_MAX_TRANSFORMS", "0"))

_HERE = Path.cwd()
_REPO_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE
OUTPUT_DIR = Path(
    os.environ.get(
        "SABENCH_GRID_OUTPUT_DIR",
        str(_REPO_ROOT / "outputs" / "noncommutativity_grid_analysis"),
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_keys = tuple(BENCHMARK_REGISTRY)
transform_keys = tuple(TRANSFORM_REGISTRY)
if MAX_BENCHMARKS > 0:
    benchmark_keys = benchmark_keys[:MAX_BENCHMARKS]
if MAX_TRANSFORMS > 0:
    transform_keys = transform_keys[:MAX_TRANSFORMS]

print(
    f"Grid: {len(benchmark_keys)} benchmarks × {len(transform_keys)} transforms, "
    f"N_BASE={N_BASE}, TAU={TAU}, seed={RNG_SEED}."
)

## Execute the registry-driven grid

Each result row represents one benchmark/transformation pair. Pair-level failures
are retained as status rows rather than stopping the full grid.

In [ ]:
results = evaluate_noncommutativity_grid(
    benchmark_keys=benchmark_keys,
    transform_keys=transform_keys,
    n_base=N_BASE,
    seed=RNG_SEED,
    tau=TAU,
    top_k=TOP_K,
)
rows = [result.as_dict() for result in results]
df = pd.DataFrame(rows)

# Normalize column names to short forms used in exports and summaries
_RENAME = {
    "decision_score_s1": "D_s1",
    "decision_score_st": "D_st",
    "sensitivity_shift_s1": "delta_s1",
    "sensitivity_shift_st": "delta_st",
    "threshold_flip_count_s1": "threshold_flip_s1",
    "threshold_flip_count_st": "threshold_flip_st",
    "top_source_index_s1": "top_driver_y_s1",
    "top_source_index_st": "top_driver_y_st",
    "top_transformed_index_s1": "top_driver_z_s1",
    "top_transformed_index_st": "top_driver_z_st",
}
df = df.rename(columns={k: v for k, v in _RENAME.items() if k in df.columns})

print(f"Evaluated {len(df)} candidate pairs")
print(df["metrics_status"].value_counts().to_string())

## Export pair-level tables

The exported tables are the durable artifacts for downstream review and plotting.
`pair_status.csv` contains every candidate pair. `noncommutativity_metrics.csv`
contains only rows with computed Sobol-profile metrics.

In [ ]:
PAIR_STATUS_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "pair_status",
    "metrics_status",
    "reason",
    "benchmark_output_kind",
    "transform_mechanism",
    "transform_tags",
    "n_base",
    "seed",
    "n_inputs",
    "n_evaluations",
    "raw_output_shape",
    "transformed_output_shape",
    "raw_output_finite",
    "transformed_output_finite",
    "raw_variance",
    "transformed_variance",
]
METRIC_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "D_s1",
    "delta_s1",
    "D_st",
    "delta_st",
    "threshold_flip_s1",
    "threshold_flip_st",
    "topk_changed_s1",
    "topk_changed_st",
    "max_abs_shift_s1",
    "max_abs_shift_st",
    "mean_abs_shift_s1",
    "mean_abs_shift_st",
    "spearman_s1",
    "spearman_st",
    "top_driver_y_s1",
    "top_driver_z_s1",
    "top_driver_y_st",
    "top_driver_z_st",
]

df_status = df[[c for c in PAIR_STATUS_COLUMNS if c in df.columns]].copy()
df_metrics = df.loc[df["metrics_status"] == "computed", [c for c in METRIC_COLUMNS if c in df.columns]].reset_index(drop=True)

df_status.to_csv(OUTPUT_DIR / "pair_status.csv", index=False)
df_metrics.to_csv(OUTPUT_DIR / "noncommutativity_metrics.csv", index=False)

print(f"Wrote pair_status.csv ({len(df_status)} rows)")
print(f"Wrote noncommutativity_metrics.csv ({len(df_metrics)} rows)")
try:
    display(df_metrics.head(10)) if len(df_metrics) > 0 else print("No computed metric rows.")
except NameError:
    print(df_metrics.head(10).to_string()) if len(df_metrics) > 0 else print("No computed metric rows.")

## Summaries by transform and benchmark

These summaries rank transformations and benchmarks by the empirical size of the
Sobol-profile perturbation.

In [ ]:
SUMMARY_METRICS = ["D_s1", "delta_s1", "D_st", "delta_st"]

if not df_metrics.empty:
    available = [m for m in SUMMARY_METRICS if m in df_metrics.columns]

    df_by_transform = (
        df_metrics.groupby("transform_key")[available]
        .agg(["mean", "max"])
        .round(4)
    )
    df_by_transform.columns = [f"{agg}_{m}" for m, agg in df_by_transform.columns]
    df_by_transform = df_by_transform.reset_index()

    df_by_benchmark = (
        df_metrics.groupby("benchmark_key")[available]
        .agg(["mean", "max"])
        .round(4)
    )
    df_by_benchmark.columns = [f"{agg}_{m}" for m, agg in df_by_benchmark.columns]
    df_by_benchmark = df_by_benchmark.reset_index()

    df_by_transform.to_csv(OUTPUT_DIR / "summary_by_transform.csv", index=False)
    df_by_benchmark.to_csv(OUTPUT_DIR / "summary_by_benchmark.csv", index=False)

    print(f"Wrote summary_by_transform.csv ({len(df_by_transform)} rows)")
    print(f"Wrote summary_by_benchmark.csv ({len(df_by_benchmark)} rows)")
    print("\nTop 10 pairs by first-order sensitivity shift (delta_s1):")
    if "delta_s1" in df_metrics.columns:
        try:
            display(
                df_metrics.nlargest(10, "delta_s1")[
                    ["benchmark_key", "transform_key", "D_s1", "delta_s1", "D_st", "delta_st"]
                ].reset_index(drop=True)
            )
        except NameError:
            print(
                df_metrics.nlargest(10, "delta_s1")[
                    ["benchmark_key", "transform_key", "D_s1", "delta_s1", "D_st", "delta_st"]
                ].reset_index(drop=True).to_string()
            )
else:
    df_by_transform = pd.DataFrame(columns=["transform_key"])
    df_by_benchmark = pd.DataFrame(columns=["benchmark_key"])
    df_by_transform.to_csv(OUTPUT_DIR / "summary_by_transform.csv", index=False)
    df_by_benchmark.to_csv(OUTPUT_DIR / "summary_by_benchmark.csv", index=False)
    print("No computed pairs — summary tables written as empty.")

## Quick interpretation checks

The output tables should be interpreted as empirical diagnostics. The Decision
Score `D` captures soft threshold-crossing behavior near `tau`, while Sensitivity
Shift `delta` captures Bray-Curtis profile movement. A large value in either
metric indicates that the transformation materially changed the benchmark's
estimated Sobol sensitivity profile.

In [ ]:
print("Pair status breakdown:")
print(df["pair_status"].value_counts().to_string())
print()
if not df_metrics.empty and "delta_s1" in df_metrics.columns:
    top = df_metrics.nlargest(1, "delta_s1").iloc[0]
    print(f"Largest first-order shift: {top['benchmark_key']} + {top['transform_key']}  delta_s1={top['delta_s1']:.4f}")
if not df_metrics.empty and "delta_st" in df_metrics.columns:
    top_st = df_metrics.nlargest(1, "delta_st").iloc[0]
    print(f"Largest total-effect shift: {top_st['benchmark_key']} + {top_st['transform_key']}  delta_st={top_st['delta_st']:.4f}")